In [32]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/sports.db")

games = pd.read_sql("""
    SELECT g.id, g.season, g.date,
           h.name AS home, a.name AS away,
           g.home_score, g.away_score,
           x.home_xg, x.away_xg
    FROM games g
    JOIN teams h ON h.id = g.home_team_id
    JOIN teams a ON a.id = g.away_team_id
    LEFT JOIN game_xg x ON x.game_id = g.id AND x.source = 'understat'
    WHERE g.status = 'final'
    ORDER BY g.date
""", conn)

games.tail()
   

,id,season,date,home,away,home_score,away_score,home_xg,away_xg
12749,12750,2026,2026-09-19,Nott'm Forest,Coventry,0,1,1.318520,0.69375
12750,12751,2026,2026-09-20,Bournemouth,Liverpool,0,1,0.944706,1.29116
12751,12752,2026,2026-09-20,Leeds,Crystal Palace,0,0,1.085520,1.12905
12752,12753,2026,2026-09-20,Man City,Sunderland,5,3,2.668560,3.10661
12753,12754,2026,2026-09-20,Fulham,Man United,1,1,1.717190,2.12453


In [ ]:
def fit_poisson(train, as_of, half_life_days=365, shrinkage=1.0, xg_weight=0.5):
    teams = sorted(set(train["home"]) | set(train["away"]))
    index = {team: i for i, team in enumerate(teams)}
    n = len(teams)

    home_idx = train["home"].map(index).to_numpy()
    away_idx = train["away"].map(index).to_numpy()
        # Blend actual goals with xG. Matches without xG (before 2014/15) fall back to goals.
    home_xg = train["home_xg"].fillna(train["home_score"])
    away_xg = train["away_xg"].fillna(train["away_score"])
    home_goals = ((1 - xg_weight) * train["home_score"] + xg_weight * home_xg).to_numpy()
    away_goals = ((1 - xg_weight) * train["away_score"] + xg_weight * away_xg).to_numpy()

    days_ago = (pd.Timestamp(as_of) - pd.to_datetime(train["date"])).dt.days.to_numpy()
    weights = 0.5 ** (days_ago / half_life_days)

    def unpack(params):
        attack = params[:n] - params[:n].mean()
        defense = params[n:2 * n] - params[n:2 * n].mean()
        return attack, defense, params[2 * n], params[2 * n + 1]

    def objective(params):
        attack, defense, base, home_adv = unpack(params)
        log_home = base + home_adv + attack[home_idx] - defense[away_idx]
        log_away = base + attack[away_idx] - defense[home_idx]
        exp_home, exp_away = np.exp(log_home), np.exp(log_away)

        # Poisson log-likelihood, minus a constant that doesn't depend on the ratings
        log_lik = home_goals * log_home - exp_home + away_goals * log_away - exp_away
        value = -np.sum(weights * log_lik) + shrinkage * np.sum(attack**2 + defense**2)

        # Gradient: how the value changes as each rating changes
        res_home = weights * (exp_home - home_goals)
        res_away = weights * (exp_away - away_goals)
        grad_attack = (np.bincount(home_idx, res_home, n) + np.bincount(away_idx, res_away, n)
                       + 2 * shrinkage * attack)
        grad_defense = (-np.bincount(away_idx, res_home, n) - np.bincount(home_idx, res_away, n)
                        + 2 * shrinkage * defense)
        grad = np.concatenate([
            grad_attack - grad_attack.mean(),    # account for the centering in unpack()
            grad_defense - grad_defense.mean(),
            [res_home.sum() + res_away.sum(), res_home.sum()],
        ])
        return value, grad

    result = minimize(objective, np.zeros(2 * n + 2), jac=True, method="L-BFGS-B")
    if not result.success:
        print("Warning: optimizer did not converge:", result.message)

    attack, defense, base, home_adv = unpack(result.x)
    return {
        "ratings": pd.DataFrame({"attack": np.exp(attack), "defense": np.exp(defense)}, index=teams),
        "base": np.exp(base),
        "home_adv": np.exp(home_adv),
    }

In [34]:
as_of = games["date"].max()
train = games[games["date"] >= "2023-07-01"]
model = fit_poisson(train, as_of)

print(f"Base expected goals: {model['base']:.2f}")
print(f"Home advantage: x{model['home_adv']:.2f}")

current_teams = games.loc[games["season"] == 2026, "home"].unique()
model["ratings"].loc[current_teams].sort_values("attack", ascending=False).round(2)

Base expected goals: 1.23
Home advantage: x1.18


,attack,defense
Man City,1.59,1.46
Arsenal,1.41,1.83
Liverpool,1.41,1.19
Chelsea,1.30,1.02
Newcastle,1.29,1.01
Brighton,1.28,1.11
Man United,1.24,1.07
Brentford,1.21,1.08
Aston Villa,1.17,1.06
Bournemouth,1.15,1.08


In [35]:
def predict(model, home, away, max_goals=10):
    r = model["ratings"]
    exp_home = model["base"] * model["home_adv"] * r.loc[home, "attack"] / r.loc[away, "defense"]
    exp_away = model["base"] * r.loc[away, "attack"] / r.loc[home, "defense"]

    goals = np.arange(max_goals + 1)
    grid = np.outer(poisson.pmf(goals, exp_home), poisson.pmf(goals, exp_away))

    return {
        "exp_home": exp_home,
        "exp_away": exp_away,
        "home": np.tril(grid, -1).sum(),
        "draw": np.trace(grid),
        "away": np.triu(grid, 1).sum(),
        "grid": grid,
    }

In [36]:
p = predict(model, "Arsenal", "Man City")

print(f"Expected goals: Arsenal {p['exp_home']:.2f} - {p['exp_away']:.2f} Man City")
print(f"Home win {p['home']:.1%}   Draw {p['draw']:.1%}   Away win {p['away']:.1%}")
print(f"Fair odds: {1/p['home']:.2f} / {1/p['draw']:.2f} / {1/p['away']:.2f}")

pd.DataFrame(
    p["grid"][:5, :5] * 100,
    index=pd.Index(range(5), name="Arsenal goals"),
    columns=pd.Index(range(5), name="Man City goals"),
).round(1)

Expected goals: Arsenal 1.41 - 1.07 Man City
Home win 44.8%   Draw 26.6%   Away win 28.6%
Fair odds: 2.23 / 3.76 / 3.49


Man City goals,0,1,2,3,4
Arsenal goals,,,,,
0,8.3,8.9,4.8,1.7,0.5
1,11.7,12.6,6.8,2.4,0.7
2,8.3,8.9,4.8,1.7,0.5
3,3.9,4.2,2.3,0.8,0.2
4,1.4,1.5,0.8,0.3,0.1


In [37]:
market = pd.read_sql("""
    WITH closing AS (
        SELECT game_id, bookmaker,
               MAX(CASE WHEN outcome = 'home' THEN 1.0 / price END) AS h,
               MAX(CASE WHEN outcome = 'draw' THEN 1.0 / price END) AS d,
               MAX(CASE WHEN outcome = 'away' THEN 1.0 / price END) AS a
        FROM odds
        WHERE market = '1x2' AND is_closing = 1
          AND bookmaker IN ('Pinnacle', 'Betfair Exchange')
        GROUP BY game_id, bookmaker
        HAVING COUNT(*) = 3
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (
                      PARTITION BY game_id
                      ORDER BY bookmaker = 'Pinnacle' DESC) AS rn
        FROM closing
    )
    SELECT game_id AS id, bookmaker AS benchmark,
           h / (h + d + a) AS mkt_home,
           d / (h + d + a) AS mkt_draw,
           a / (h + d + a) AS mkt_away
    FROM ranked
    WHERE rn = 1
""", conn)

market["benchmark"].value_counts()

benchmark
Pinnacle            5150
Betfair Exchange     198
Name: count, dtype: int64

In [38]:
def backtest(games, start_season, **model_settings):
    test = games[games["season"] >= start_season].copy()
    test["week"] = pd.to_datetime(test["date"]).dt.to_period("W").dt.start_time

    rows = []
    for week_start, week_games in test.groupby("week"):
        cutoff = week_start.strftime("%Y-%m-%d")
        window_start = (week_start - pd.DateOffset(years=3)).strftime("%Y-%m-%d")
        train = games[(games["date"] >= window_start) & (games["date"] < cutoff)]
        model = fit_poisson(train, cutoff, **model_settings)
        known = set(model["ratings"].index)

        for g in week_games.itertuples():
            if g.home not in known or g.away not in known:
                continue  # a team with no games in the training window yet
            p = predict(model, g.home, g.away)
            if g.home_score > g.away_score:
                result = "home"
            elif g.home_score == g.away_score:
                result = "draw"
            else:
                result = "away"
            rows.append({"id": g.id, "season": g.season, "result": result,
                         "model_home": p["home"], "model_draw": p["draw"], "model_away": p["away"]})
    return pd.DataFrame(rows)

results = backtest(games, start_season=2016)
len(results)

3833

In [39]:
def add_scores(df, prefix):
    outcomes = ["home", "draw", "away"]
    probs = df[[f"{prefix}_{o}" for o in outcomes]].to_numpy()
    actual = (df["result"].to_numpy()[:, None] == np.array(outcomes)).astype(float)
    df[f"{prefix}_logloss"] = -np.log((probs * actual).sum(axis=1))
    df[f"{prefix}_brier"] = ((probs - actual) ** 2).sum(axis=1)

scored = results.merge(market, on="id")
add_scores(scored, "model")
add_scores(scored, "mkt")

summary = scored.groupby("season")[["model_logloss", "mkt_logloss", "model_brier", "mkt_brier"]].mean()
summary.insert(0, "games", scored.groupby("season").size())
print(scored[["model_logloss", "mkt_logloss", "model_brier", "mkt_brier"]].mean().round(4))
summary.round(4)

model_logloss    0.9693
mkt_logloss      0.9507
model_brier      0.5753
mkt_brier        0.5628
dtype: float64


,games,model_logloss,mkt_logloss,model_brier,mkt_brier
season,,,,,
2016,379,0.9384,0.9056,0.5544,0.5317
2017,378,0.9612,0.9402,0.5709,0.5568
2018,377,0.8895,0.8900,0.5196,0.5203
2019,377,0.9776,0.9750,0.5812,0.5763
2020,379,1.0184,0.9990,0.6060,0.5935
2021,379,0.9535,0.9355,0.5645,0.5535
2022,379,1.0006,0.9634,0.5969,0.5721
2023,379,0.9293,0.9012,0.5473,0.5270
2024,379,0.9853,0.9681,0.5889,0.5763


In [40]:
tuning_games = games[games["season"] <= 2020]

rows = []
for half_life in [90, 180, 365, 730]:
    for shrinkage in [0.5, 1, 3, 10]:
        res = backtest(tuning_games, start_season=2016,
                       half_life_days=half_life, shrinkage=shrinkage)
        add_scores(res, "model")
        rows.append({"half_life": half_life, "shrinkage": shrinkage,
                     "logloss": res["model_logloss"].mean()})

tuning = pd.DataFrame(rows)
tuning.pivot(index="half_life", columns="shrinkage", values="logloss").round(4)

shrinkage,0.5,1.0,3.0,10.0
half_life,,,,
90,0.9671,0.9653,0.9658,0.9826
180,0.9584,0.9577,0.9584,0.9689
365,0.9577,0.9571,0.9572,0.9632
730,0.9594,0.9588,0.9586,0.9624


In [41]:
tuned = backtest(games, start_season=2021)
old = backtest(games, start_season=2021, half_life_days=180, shrinkage=3.0)

old = old[["id", "model_home", "model_draw", "model_away"]]
old.columns = ["id", "old_home", "old_draw", "old_away"]

test = tuned.merge(market, on="id").merge(old, on="id")
for prefix in ["model", "old", "mkt"]:
    add_scores(test, prefix)

print(f"Games: {len(test)}")
test[["old_logloss", "model_logloss", "mkt_logloss",
      "old_brier", "model_brier", "mkt_brier"]].mean().round(4)

Games: 1921


old_logloss      0.9824
model_logloss    0.9813
mkt_logloss      0.9592
old_brier        0.5847
model_brier      0.5840
mkt_brier        0.5697
dtype: float64

In [42]:
tuning_games = games[games["season"] <= 2020]

rows = []
for xg_weight in [0, 0.25, 0.5, 0.75, 1.0]:
    res = backtest(tuning_games, start_season=2016, xg_weight=xg_weight)
    add_scores(res, "model")
    rows.append({"xg_weight": xg_weight, "logloss": res["model_logloss"].mean()})

pd.DataFrame(rows).round(4)

,xg_weight,logloss
0,0.00,0.9571
1,0.25,0.9544
2,0.50,0.9532
3,0.75,0.9535
4,1.00,0.9552
